# LibRA deployment smoke test

Container-level check that the LibRA wheel is installed and working, using a real EVLA MS (`SNR_G55_10s.calib.ms`, bundled at build time from `casa.nrao.edu`).

**Cell 1** runs `TableInfo` against the MS and prints what's in it -- this is the smoke test. Get this working before touching the imaging cells below.

In [ ]:
import os
import libra

# Casacore's compiled-in measures directory won't exist in this container --
# point it at the data bundled inside the wheel instead.
data_path = libra.get_data_path()
casarc_path = os.path.expanduser("~/.casarc")
with open(casarc_path, "w") as f:
    f.write(f"measures.directory: {data_path}\n")

print(f"libra {libra.__version__}")
print(f"data path: {data_path}")

In [ ]:
vis = os.path.expanduser("~/sample_data/SNR_G55_10s.calib.ms")

ti = libra.TableInfo(MSNBuf=vis, verbose=True).run()
print(ti)

## Imaging pass (fill in before running)

Cell 1 above is the container smoke test. The cells below run an actual `roadrunner` -> `hummbee`/`dale` -> `restore` imaging pass, CPU-only (`HPGDEVICE=libra_omp`), but need MS-specific values that shouldn't be guessed:

- `cell_size` / `NX` (imsize) for `Coyote`'s convolution-function cache -- get these from the MS's field/spw geometry, not made up.
- `refFreqStr` -- SNR G55.10 was observed with the EVLA; pull the actual reference frequency from `TableInfo`'s spectral window summary above.
- The `Restore` step needs the PSF's fitted beam (`majaxis`, `minaxis`, `pa`) -- fit this from the `Dale`-normalized PSF image `roadrunner` produces, don't hardcode a guess.

The call sequence itself is correct (matches the `libra-python` wrapper API); the parameter values below are placeholders.

In [ ]:
os.environ["HPGDEVICE"] = "libra_omp"  # CPU-only Kokkos backend

imagename = os.path.expanduser("~/sample_data/snr_g55_demo")
cfcache = os.path.expanduser("~/sample_data/snr_g55.cf")

# TODO: fill in from TableInfo's field/spw output above -- do not run as-is
cell_size = None   # e.g. "1.0arcsec"
NX = None          # e.g. 2048
ref_freq = None    # e.g. "3GHz", from the spw summary
phasecenter = ""   # optional; defaults to the MS's own phase center

assert cell_size and NX and ref_freq, "fill in cell_size / NX / ref_freq from TableInfo output first"

In [ ]:
# Step 1: convolution-function cache (required before roadrunner's awphpg gridder)
coyote = libra.Coyote(
    MSNBuf=vis,
    telescopeName="EVLA",
    NX=NX,
    cellSize=cell_size,
    stokes="I",
    refFreqStr=ref_freq,
    cfCacheName=cfcache,
    phaseCenter=phasecenter,
).run()
print(coyote)

In [ ]:
# Step 2: grid weight, psf, residual (roadrunner's mode="residual" default does psf+residual)
rr = libra.RoadRunner(
    vis=vis,
    imagename=imagename,
    imsize=[NX, NX],
    cell=[cell_size, cell_size],
    reffreq=ref_freq,
    phasecenter=phasecenter,
    cfcache=cfcache,
    mode="residual",
).run()
print(rr)

In [ ]:
# Step 3: normalize the PSF (flat-noise) before deconvolving
dale = libra.Dale(
    imageName=f"{imagename}.psf",
    wtimageName=f"{imagename}.weight",
    normtype="flatnoise",
    imType="psf",
).run()
print(dale)

In [ ]:
# Step 4: deconvolve
hb = libra.Hummbee(
    imagename=imagename,
    deconvolver="hogbom",
    threshold=0.0,
    cycleniter=-1,
    mode="deconvolve",
).run()
print(hb)

In [ ]:
# Step 5: restore -- fit majaxis/minaxis/pa from the PSF before running this;
# do not use the placeholder zeros below.
restore = libra.Restore(
    model=f"{imagename}.model",
    residual=f"{imagename}.residual",
    image=f"{imagename}.image",
    size_x=NX,
    size_y=NX,
    majaxis=0.0,  # TODO: fit from PSF
    minaxis=0.0,  # TODO: fit from PSF
    pa=0.0,       # TODO: fit from PSF
).run()
print(restore)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from libra import GetChunk

img = GetChunk(imageName=f"{imagename}.image", type="image").run()
plt.imshow(np.squeeze(img.result), origin="lower", cmap="inferno")
plt.colorbar(label="Jy/beam")
plt.title("SNR G55.10 -- LibRA CPU demo")